# Guia Definitivo -- Cientista de Dados Itau Unibanco
## ISLP + Estatistica + ML + Pegadinhas da Prova

**Objetivo:** Consolidar todo o conhecimento para passar na sabatina  
**Base:** ISLP (James, Witten, Hastie, Tibshirani, Taylor) + prova real 2019  
**Abordagem:** Conceito -> Formula -> Codigo -> Pegadinha da prova

---

## Estrutura

| Modulo | Tema | Cap. ISLP |
|--------|------|-----------|
| 0 | Setup | -- |
| 1 | Estatistica Basica | Cap. 1 + 13 |
| 2 | Aprendizado Estatistico | Cap. 2 |
| 3 | Regressao Linear | Cap. 3 |
| 4 | Classificacao e Logistica | Cap. 4 |
| 5 | Cross-Validation | Cap. 5 |
| 6 | Regularizacao Ridge, Lasso, ElasticNet | Cap. 6 |
| 7 | Alem da Linearidade | Cap. 7 |
| 8 | Arvores, Random Forest, XGBoost | Cap. 8 |
| 9 | SVM | Cap. 9 |
| 10 | Redes Neurais | Cap. 10 |
| 11 | PCA + Clustering | Cap. 12 |
| 12 | MLflow | -- |
| 13 | Simulado Completo da Prova | -- |
| 14 | Cola Rapida e Treino de Velocidade | -- |


---
# MODULO 0 -- Setup

Execute esta celula antes de qualquer outra.

In [ ]:
!pip install numpy pandas scikit-learn matplotlib seaborn scipy xgboost mlflow statsmodels --quiet
print("Dependencias instaladas!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings, os, math
warnings.filterwarnings('ignore')

from sklearn.linear_model import (LinearRegression, LogisticRegression,
                                   Ridge, Lasso, ElasticNet)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, SVR
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import (train_test_split, KFold, StratifiedKFold,
                                      TimeSeriesSplit, cross_validate,
                                      learning_curve)
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              log_loss, confusion_matrix, roc_auc_score,
                              roc_curve, precision_recall_curve, f1_score,
                              silhouette_score)
from sklearn.cluster import KMeans
from sklearn.datasets import make_classification
from sklearn.preprocessing import SplineTransformer

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from xgboost import XGBClassifier
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import mlflow, mlflow.sklearn

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print("Imports OK!")

---
# MODULO 1 -- Estatistica Basica
## ISLP Capitulos 1 e 13

> Base para TUDO que vem depois.  
> Se voce nao entende p-valor, nao entende por que um coeficiente importa.

---

## 1.1 Tipos de Variaveis

| Tipo | Subtipos | Exemplos |
|------|----------|----------|
| Quantitativa | Continua / Discreta | NPS (0-10), n transacoes |
| Qualitativa | Nominal / Ordinal | Segmento PJ, rating |

## 1.2 Medidas de tendencia e dispersao

| Medida | Sensivel a outliers? | Quando usar |
|--------|----------------------|-------------|
| Media | Sim | Dados simetricos |
| Mediana | Nao | Dados financeiros com outliers |
| Desvio padrao | Sim | Dispersao em escala original |
| IQR | Nao | Dispersao robusta |

## 1.3 Distribuicoes de Probabilidade

| Distribuicao | Range | Uso tipico em banco |
|-------------|-------|---------------------|
| Normal | (-inf, +inf) | Erros de regressao linear |
| Bernoulli | {0, 1} | Y binario na logistica |
| Binomial | {0,...,n} | N de inadimplentes em n clientes |
| Poisson | {0,1,2,...} | N de transacoes por dia |
| Log-normal | (0, +inf) | Volume financeiro, renda |

**Pegadinha da prova (Q12):**  
'Erros da Regressao Logistica devem ter distribuicao Normal' -> FALSO  
Logistica assume Bernoulli para Y. Normal e pressuposto da REGRESSAO LINEAR.


In [ ]:
# 1.1 -- Tipos de variaveis e estatisticas descritivas
np.random.seed(42)
n = 500

dados = pd.DataFrame({
    'nps_nota':       np.random.choice(range(0,11), n,
                       p=[0.02,0.02,0.03,0.04,0.05,0.06,0.08,0.10,0.15,0.20,0.25]),
    'tempo_conta':    np.random.exponential(5, n),
    'volume_transac': np.random.lognormal(8, 1.5, n),
    'segmento':       np.random.choice(['PME','Medio','Large'], n, p=[0.6,0.3,0.1]),
})
dados['categoria_nps'] = dados['nps_nota'].apply(
    lambda x: 'Detrator' if x<=6 else ('Neutro' if x<=8 else 'Promotor'))

print("=== ESTATISTICAS DESCRITIVAS ===")
print(dados[['nps_nota','tempo_conta','volume_transac']].describe().round(2))
print()
print("Distribuicao NPS:")
print(dados['categoria_nps'].value_counts())

In [ ]:
# 1.2 -- Media vs Mediana: impacto dos outliers
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(dados['nps_nota'], bins=11, edgecolor='black', color='steelblue')
axes[0].axvline(dados['nps_nota'].mean(), color='red', linestyle='--',
                label=f"Media: {dados['nps_nota'].mean():.2f}")
axes[0].axvline(dados['nps_nota'].median(), color='orange', linestyle='--',
                label=f"Mediana: {dados['nps_nota'].median():.1f}")
axes[0].set_title('NPS -- mais simetrico')
axes[0].legend()

axes[1].hist(dados['volume_transac'], bins=60, edgecolor='black', color='coral')
axes[1].axvline(dados['volume_transac'].mean(), color='red', linestyle='--',
                label=f"Media: R${dados['volume_transac'].mean():,.0f}")
axes[1].axvline(dados['volume_transac'].median(), color='orange', linestyle='--',
                label=f"Mediana: R${dados['volume_transac'].median():,.0f}")
axes[1].set_title('Volume -- assimetrico (Media >> Mediana)')
axes[1].legend()
plt.tight_layout()
plt.show()

print("REGRA: dados financeiros (renda, volume) -> use MEDIANA, nao media")

In [ ]:
# 1.3 -- Distribuicoes de probabilidade
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
x_cont = np.linspace(-4, 4, 300)

axes[0,0].plot(x_cont, stats.norm.pdf(x_cont, 0, 1), 'b-', linewidth=2)
axes[0,0].fill_between(x_cont, stats.norm.pdf(x_cont, 0, 1), alpha=0.3)
axes[0,0].set_title('Normal(mu=0, sigma=1)\nErros de regressao linear')

p_bern = 0.3
axes[0,1].bar([0,1], [1-p_bern, p_bern], color=['steelblue','coral'], edgecolor='black')
axes[0,1].set_xticks([0,1])
axes[0,1].set_xticklabels(['nao inadimpliu','inadimpliu'])
axes[0,1].set_title(f'Bernoulli(p={p_bern})\nErros de regressao logistica')

k_bin = np.arange(0, 21)
axes[0,2].bar(k_bin, stats.binom.pmf(k_bin, 20, 0.3), color='steelblue', edgecolor='black')
axes[0,2].set_title('Binomial(n=20, p=0.3)\nN de inadimplentes em 20 clientes')

k_pois = np.arange(0, 20)
axes[1,0].bar(k_pois, stats.poisson.pmf(k_pois, 5), color='green', edgecolor='black')
axes[1,0].set_title('Poisson(lambda=5)\nN de transacoes por dia')

x_log = np.linspace(0.01, 10, 300)
axes[1,1].plot(x_log, stats.lognorm.pdf(x_log, s=1, scale=np.exp(1)), 'r-', linewidth=2)
axes[1,1].fill_between(x_log, stats.lognorm.pdf(x_log, s=1, scale=np.exp(1)), alpha=0.3)
axes[1,1].set_title('Log-normal\nVolume financeiro, renda')

axes[1,2].plot(x_cont, stats.norm.pdf(x_cont), 'b-', linewidth=2, label='Normal')
axes[1,2].plot(x_cont, stats.t.pdf(x_cont, df=3), 'r-', linewidth=2, label='t(df=3) cauda pesada')
axes[1,2].set_title('Normal vs t-Student\nDados financeiros: caudas pesadas')
axes[1,2].legend()
plt.tight_layout()
plt.show()

## 1.4 Inferencia Estatistica -- Testes de Hipotese

### P-valor -- definicao exata (cobrada na prova):
> Probabilidade de observar um resultado tao extremo quanto o observado,  
> **assumindo que H0 e verdadeira**

**ERRO COMUM:** 'p-valor = probabilidade de H0 ser verdadeira' -> ERRADO  
**CORRETO:** 'p-valor baixo = resultado improvavel se H0 fosse verdade'

### Erros tipo 1 e 2:

| | H0 verdadeira | H0 falsa |
|--|--------------|----------|
| **Rejeitar H0** | ERRO Tipo 1 (falso positivo) | Correto |
| **Nao rejeitar H0** | Correto | ERRO Tipo 2 (falso negativo) |

**Contexto banco:**
- Fraude: Erro Tipo 2 e catastrofico (deixar fraude passar)  
- Campanha: Erro Tipo 1 e caro (investir em algo sem efeito)

### Pearson vs Spearman:
- **Pearson:** correlacao LINEAR (-1 a 1)
- **Spearman:** correlacao MONOTONICA (nao exige linearidade)
- **NUNCA:** correlacao implica causalidade!


In [ ]:
# 1.4 -- Teste de hipotese aplicado ao contexto bancario
np.random.seed(42)
nps_antes  = np.random.normal(7.2, 1.5, 100)
nps_depois = np.random.normal(7.6, 1.4, 100)
t_stat, p_valor = stats.ttest_ind(nps_depois, nps_antes)

print("=== TESTE: nova regua de atendimento melhorou NPS? ===")
print(f"H0: media_depois = media_antes (sem efeito)")
print(f"H1: media_depois > media_antes")
print(f"Media antes:   {nps_antes.mean():.3f}")
print(f"Media depois:  {nps_depois.mean():.3f}")
print(f"Estatistica t: {t_stat:.4f}")
print(f"P-valor:       {p_valor:.4f}")
print()
if p_valor < 0.05:
    print("p < 0.05 -> Rejeitamos H0: evidencia de melhora")
else:
    print("p >= 0.05 -> Nao rejeitamos H0: sem evidencia suficiente")

diff = nps_depois.mean() - nps_antes.mean()
se   = np.sqrt(nps_depois.var()/len(nps_depois) + nps_antes.var()/len(nps_antes))
ic   = (diff - 1.96*se, diff + 1.96*se)
print(f"IC 95%: [{ic[0]:.3f}, {ic[1]:.3f}]")

print()
print("PEGADINHAS DO P-VALOR:")
print("  ERRADO:  p=0.03 = 97% de chance do efeito ser real")
print("  CORRETO: p=0.03 = resultado seria raro se H0 fosse verdade")
print()
print("CORRELACAO vs CAUSALIDADE:")
corr_ex, _ = stats.pearsonr(nps_antes, np.random.normal(5, 1, 100))
print(f"  Correlacao pode ser espuria (confundidor oculto)")
print(f"  Para causalidade: experimento randomizado ou quasi-experimental")

---
# MODULO 2 -- Aprendizado Estatistico
## ISLP Capitulo 2

> "Statistical learning refers to a set of tools for making sense of complex datasets."

## 2.1 O problema fundamental

```
Y = f(X) + epsilon
```

| Elemento | Exemplo (Itau) |
|----------|----------------|
| Y | NPS, inadimplencia, churn |
| X | Tempo de atendimento, segmento, produto |
| f(X) | O que o modelo aprende |
| epsilon | Erro irredutivel -- nenhum modelo elimina |

## 2.2 Por que estimamos f? Duas razoes:

| Objetivo | Modelo preferido |
|----------|-----------------|
| Predicao (prever Y) | Qualquer modelo preciso |
| Inferencia (entender X->Y) | Modelo interpretavel |

## 2.3 Bias-Variance Tradeoff

```
Erro total = Bias^2 + Variancia + Erro irredutivel
```

- **Alto Bias (underfitting):** modelo simples demais, erra sistematicamente  
- **Alta Variancia (overfitting):** modelo memoriza ruido, falha em dados novos  
- **Objetivo:** encontrar o equilibrio

## 2.4 Trade-off Predicao vs Interpretabilidade (ISLP Fig. 2.7)

```
MAIS FLEXIVEL
    Redes Neurais Profundas  <- melhor predicao, caixa preta
    XGBoost / Gradient Boosting
    Random Forest
    Arvores de Decisao
    SVM com kernel RBF
    Regressao Logistica
    Regressao Linear  <- mais interpretavel
MENOS FLEXIVEL
```

Em ambiente regulado (BACEN): interpretabilidade e exigencia!

## 2.5 Supervisionado vs Nao-Supervisionado

| | Supervisionado | Nao-supervisionado |
|--|----------------|-------------------|
| Tem Y? | Sim | Nao |
| Exemplos | Regressao, Classificacao | Clustering, PCA |

**FRONTEIRA CRITICA (Q35 da prova):**  
Clustering NAO usa rotulos -> NUNCA e supervisionado.


In [ ]:
# 2.1 -- Y = f(X) + epsilon e o erro irredutivel
np.random.seed(42)
X_ex = np.linspace(0, 10, 200).reshape(-1, 1)
f_real = 3 * np.sin(X_ex.ravel() / 2) + 0.5 * X_ex.ravel()
epsilon_ex = np.random.normal(0, 1.2, 200)
Y_ex = f_real + epsilon_ex

modelo_ex = LinearRegression().fit(X_ex, Y_ex)
Y_pred_ex = modelo_ex.predict(X_ex)

plt.figure(figsize=(10, 5))
plt.scatter(X_ex, Y_ex, alpha=0.3, s=15, label='Y observado (f + epsilon)')
plt.plot(X_ex, f_real, 'g-', linewidth=2.5, label='f(X) real (desconhecida)')
plt.plot(X_ex, Y_pred_ex, 'r-', linewidth=2.5, label='f-hat(X) estimada')
plt.legend()
plt.title('Y = f(X) + epsilon  |  O modelo aprende f, nunca elimina epsilon')
plt.tight_layout()
plt.show()

print(f"Desvio padrao do epsilon real: 1.200")
print(f"Std dos residuos:              {np.std(Y_ex - Y_pred_ex):.3f}")
print("-> O modelo nao consegue ir abaixo do erro irredutivel")

In [ ]:
# 2.2 -- Bias-Variance Tradeoff demonstrado
np.random.seed(42)
X_bv = np.sort(np.random.uniform(0, 10, 100)).reshape(-1, 1)
Y_bv = np.sin(X_bv.ravel()) + np.random.normal(0, 0.3, 100)
X_plot_bv = np.linspace(0, 10, 300).reshape(-1, 1)
kf_bv = KFold(n_splits=5, shuffle=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, grau, titulo in zip(axes,
    [1, 3, 12],
    ['Grau 1 -- ALTO BIAS\n(underfitting)',
     'Grau 3 -- EQUILIBRIO\n(ponto otimo)',
     'Grau 12 -- ALTA VARIANCIA\n(overfitting)']):

    pipe = Pipeline([('poly', PolynomialFeatures(grau)), ('lr', LinearRegression())])
    pipe.fit(X_bv, Y_bv)
    mse_tr = np.mean((Y_bv - pipe.predict(X_bv))**2)
    mse_va = -cross_validate(pipe, X_bv, Y_bv, cv=kf_bv,
                              scoring='neg_mean_squared_error')['test_score'].mean()
    ax.scatter(X_bv, Y_bv, alpha=0.4, s=15, color='gray')
    ax.plot(X_plot_bv, pipe.predict(X_plot_bv), 'r-', linewidth=2.5)
    ax.set_title(f'{titulo}\nMSE treino:{mse_tr:.3f} | val:{mse_va:.3f}')
    ax.set_ylim(-3, 3)

plt.tight_layout()
plt.show()

print("Grau 1:  MSE treino ALTO -> underfitting, alto bias")
print("Grau 3:  MSE treino e val balanceados -> equilibrio")
print("Grau 12: MSE treino ~0, val alto -> overfitting, alta variancia")

---
# MODULO 3 -- Regressao Linear
## ISLP Capitulo 3

## Equacao
```
Y = beta_0 + beta_1*X1 + beta_2*X2 + ... + beta_p*Xp + epsilon
```

**beta_1:** quanto Y muda para cada unidade de X1, **mantendo demais constantes**

## Premissas (ISLP 3.3.3):
1. Linearidade
2. Independencia das observacoes
3. Homocedasticidade (variancia constante dos erros)
4. Normalidade dos residuos
5. Sem multicolinearidade

## Metricas:

| Metrica | Sensivel a outliers? | Diminui com mais vars? |
|---------|----------------------|------------------------|
| MSE | Sim | Nao |
| RMSE | Sim | Nao |
| MAE | Nao | Nao |
| R^2 | -- | **NUNCA diminui** |
| R^2 ajustado | -- | Pode diminuir |
| MAPE | -- | **INDEFINIDO se y_real=0** |

**Q15 da prova:** y = ax SEM intercepto -> media dos residuos != 0  
**Q13 da prova:** MAPE quando y_true=[0.1, 0.0, ...] -> INDEFINIDO (nao e zero!)


In [ ]:
# 3.1 -- Regressao Linear com interpretacao completa
np.random.seed(42)
n = 600
tempo_espera    = np.random.normal(8, 3, n)
tempo_resolucao = np.random.normal(15, 5, n)
n_contatos      = np.random.poisson(2.5, n)
nivel_credito   = np.random.normal(7, 1.5, n)

nps_y = (9 - 0.25*tempo_espera - 0.15*tempo_resolucao
          - 0.4*n_contatos + 0.3*nivel_credito
          + np.random.normal(0, 1, n))
nps_y = np.clip(nps_y, 0, 10)

X_cols = ['tempo_espera','tempo_resolucao','n_contatos','nivel_credito']
df_reg = pd.DataFrame(dict(zip(X_cols, [tempo_espera, tempo_resolucao, n_contatos, nivel_credito])))
df_reg['nps'] = nps_y

X_tr, X_te, y_tr, y_te = train_test_split(df_reg[X_cols], df_reg['nps'],
                                            test_size=0.2, random_state=42)
sk_model = LinearRegression().fit(X_tr, y_tr)
y_pred = sk_model.predict(X_te)

print("=== COEFICIENTES E INTERPRETACAO ===")
coefs_reais = {'tempo_espera':-0.25,'tempo_resolucao':-0.15,'n_contatos':-0.40,'nivel_credito':0.30}
for feat, coef in zip(X_cols, sk_model.coef_):
    direcao = 'SOBE' if coef > 0 else 'CAI'
    print(f"  {feat:<20}: {coef:>7.4f}  |  real: {coefs_reais[feat]:>5.2f}  "
          f"  NPS {direcao} {abs(coef):.2f} pontos por unidade")
print(f"  {'intercepto':<20}: {sk_model.intercept_:>7.4f}")

mse = mean_squared_error(y_te, y_pred)
r2  = r2_score(y_te, y_pred)
n_te, p = len(y_te), len(X_cols)
r2_adj = 1 - (1-r2)*(n_te-1)/(n_te-p-1)

print(f"\n=== METRICAS ===")
print(f"  RMSE:        {np.sqrt(mse):.4f}  (em pontos de NPS)")
print(f"  MAE:         {mean_absolute_error(y_te, y_pred):.4f}  (robusto a outliers)")
print(f"  R^2:         {r2:.4f}  -> explica {r2*100:.1f}% da variancia")
print(f"  R^2 ajust:   {r2_adj:.4f}")
print()
print("PEGADINHA: R^2 NUNCA diminui com mais variaveis!")
print("R^2 ajustado PODE diminuir -- use-o para comparar modelos")

In [ ]:
# 3.2 -- Analise de residuos e VIF
residuos = y_tr.values - sk_model.predict(X_tr)
y_fitted = sk_model.predict(X_tr)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(y_fitted, residuos, alpha=0.4, s=15)
axes[0].axhline(0, color='red', linewidth=1.5)
axes[0].set_title('Residuos vs Fitted\nSem padrao = homocedasticidade OK')

(osm, osr), (slope, intercept, r) = stats.probplot(residuos, dist='norm')
axes[1].plot(osm, osr, 'o', alpha=0.4, markersize=3)
axes[1].plot(osm, slope*np.array(osm)+intercept, 'r-', linewidth=2)
axes[1].set_title(f'QQ Plot (R={r:.3f})\nPontos na diagonal = normalidade OK')

axes[2].hist(residuos, bins=40, edgecolor='black', density=True)
x_n = np.linspace(residuos.min(), residuos.max(), 100)
axes[2].plot(x_n, stats.norm.pdf(x_n, residuos.mean(), residuos.std()), 'r-', linewidth=2)
axes[2].set_title('Distribuicao dos residuos')
plt.tight_layout(); plt.show()

print("=== VIF (Variance Inflation Factor) ===")
X_num = X_tr.values
vif_df = pd.DataFrame({'variavel': X_cols,
    'VIF': [variance_inflation_factor(X_num, i) for i in range(X_num.shape[1])]})
vif_df['status'] = vif_df['VIF'].apply(lambda v: 'OK' if v<5 else ('Atencao' if v<10 else 'PROBLEMA'))
print(vif_df.to_string(index=False))
print()
print("VIF < 5: OK | 5-10: Atencao | > 10: PROBLEMA -- remover ou combinar variaveis")
print()
print("PREMISSA CRITICA (Q15 da prova):")
print("  y = ax SEM intercepto -> media dos residuos NAO e necessariamente 0")
print("  A propriedade media=0 EXIGE intercepto no modelo")

In [ ]:
# 3.3 -- MAPE e quando nao usar
print("=== MAPE -- quando NAO usar ===")
print()
y_real_ex = np.array([10.0, 20.0, 30.0, 0.0, 50.0])
y_prev_ex = np.array([12.0, 18.0, 33.0, 2.0, 52.0])

for yr, yp in zip(y_real_ex, y_prev_ex):
    if yr == 0:
        print(f"  y_real={yr:.1f}, y_prev={yp:.1f} -> INDEFINIDO (divisao por zero!)")
    else:
        print(f"  y_real={yr:.1f}, y_prev={yp:.1f} -> MAPE = {abs(yr-yp)/abs(yr)*100:.1f}%")

print()
print("Q13 DA PROVA:")
print("  y_pred=[0.1, 0.0, 0.2, 0.1] | y_true=[0.1, 0.0, 0.2, 0.1]")
print("  Afirmar MAPE=0 esta INCORRETO -> e INDEFINIDO (y_true contem 0)")
print()
print("ALTERNATIVAS AO MAPE:")
print("  MAE: robusto a outliers, mesma unidade de Y")
print("  RMSE: penaliza erros grandes")
print("  R^2: interpretavel como % variancia explicada")

---
# MODULO 4 -- Classificacao e Regressao Logistica
## ISLP Capitulo 4

## Por que nao regressao linear para Y binario?
Regressao linear pode prever P(Y=1) < 0 ou > 1 -- sem sentido probabilistico.

## A funcao sigmoide:
```
p(X) = e^(beta_0 + beta_1*X) / (1 + e^(beta_0 + beta_1*X))
log(p/(1-p)) = beta_0 + beta_1*X1 + ... (log-odds)
```

**Odds Ratio = e^beta_1:** para cada unidade de X1, as odds se multiplicam por e^beta_1

## Todas as metricas de classificacao:

| Metrica | Formula | Quando priorizar |
|---------|---------|-----------------|
| Acuracia | (TP+TN)/total | Base balanceada |
| Precision | TP/(TP+FP) | Custo alto de FP |
| Recall/TPR | TP/(TP+FN) | Custo alto de FN (fraude!) |
| F1 | 2*P*R/(P+R) | Base desbalanceada |
| ROC-AUC | Area curva ROC | Comparacao geral |
| Log Loss | -sum(y*log(p)) | Qualidade das probabilidades |

**Pegadinhas:**
- MAE e metrica de REGRESSAO, nao classificacao (Q22!)
- Log Loss E valida para classificacao (Q20 -- candidato errou!)
- AUC > 0.95 -> suspeite de data leakage
- Acuracia enganosa com base desbalanceada


In [ ]:
# 4.1 -- Logistica vs Linear para classificacao
np.random.seed(42)
n4 = 300
score4 = np.random.normal(600, 80, n4)
inadim4 = np.clip(((score4 < 580)*1 + np.random.binomial(1, 0.1, n4)), 0, 1)
X4 = score4.reshape(-1, 1)
X_plot4 = np.linspace(350, 850, 400).reshape(-1, 1)

lin4 = LinearRegression().fit(X4, inadim4)
log4 = LogisticRegression().fit(X4, inadim4)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model, titulo, cor in zip(axes,
    [lin4, log4],
    ['Regressao LINEAR para Y binario\nPROBLEMA: valores fora de [0,1]',
     'Regressao LOGISTICA\nSempre entre 0 e 1'],
    ['red','green']):
    ax.scatter(score4, inadim4 + np.random.normal(0,0.02,n4), alpha=0.3, s=10)
    preds = model.predict_proba(X_plot4)[:,1] if hasattr(model,'predict_proba') else model.predict(X_plot4)
    ax.plot(X_plot4, preds, cor, linewidth=2.5)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axhline(1, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Score de credito'); ax.set_ylabel('P(inadimplencia)')
    ax.set_title(titulo)
plt.tight_layout(); plt.show()

print("=== ODDS RATIO -- INTERPRETACAO ===")
for b in [-1.0, -0.5, 0.3, 0.8]:
    OR = np.exp(b)
    direcao = f"caem {(1-OR)*100:.0f}%" if OR<1 else f"sobem {(OR-1)*100:.0f}%"
    print(f"  beta={b:+.1f} | OR=e^beta={OR:.4f} | odds {direcao} por unidade de X")

In [ ]:
# 4.2 -- Todas as metricas de classificacao
np.random.seed(42)
n4b = 2000
y_real4 = np.random.binomial(1, 0.2, n4b)
proba4  = np.where(y_real4==1, np.random.beta(7,3,n4b), np.random.beta(3,7,n4b))
y_pred4 = (proba4 >= 0.5).astype(int)

cm4 = confusion_matrix(y_real4, y_pred4)
TP=cm4[1,1]; TN=cm4[0,0]; FP=cm4[0,1]; FN=cm4[1,0]

print("=== MATRIZ DE CONFUSAO ===")
print(f"           | Prev 0  | Prev 1")
print(f"Real 0     | TN={TN:4d} | FP={FP:4d}   <- Nao detrator")
print(f"Real 1     | FN={FN:4d} | TP={TP:4d}   <- Detrator")
print()

metricas4 = [
    ("Acuracia", "(TP+TN)/total",       (TP+TN)/(TP+TN+FP+FN), "ATENCAO: enganosa com base desbalanceada"),
    ("Precision","TP/(TP+FP)",          TP/(TP+FP),             "dos previstos detrator, quantos sao?"),
    ("Recall",   "TP/(TP+FN)",          TP/(TP+FN),             "dos detratores reais, capturei quantos?"),
    ("F1",       "2*P*R/(P+R)",         2*(TP/(TP+FP))*(TP/(TP+FN))/((TP/(TP+FP))+(TP/(TP+FN))), "media harmonica P e R"),
    ("ROC-AUC",  "area curva ROC",      roc_auc_score(y_real4, proba4), "0.5=aleatorio, 0.8=bom, >0.95=suspeito"),
    ("Log Loss", "-sum(y*log(p))",      log_loss(y_real4, proba4),      "qualidade das probabilidades"),
]
print("=== METRICAS ===")
for nome, formula, valor, desc in metricas4:
    print(f"  {nome:<12}: {formula:<20} = {valor:.4f}  | {desc}")

print()
print("TRADEOFF DE THRESHOLD:")
print(f"{'Thresh':>8} {'Precision':>10} {'Recall':>8}")
for t in [0.3,0.4,0.5,0.6,0.7]:
    yp = (proba4>=t).astype(int)
    tp_ = ((yp==1)&(y_real4==1)).sum()
    fp_ = ((yp==1)&(y_real4==0)).sum()
    fn_ = ((yp==0)&(y_real4==1)).sum()
    p_  = tp_/(tp_+fp_) if (tp_+fp_)>0 else 0
    r_  = tp_/(tp_+fn_) if (tp_+fn_)>0 else 0
    print(f"{t:>8.1f} {p_:>10.3f} {r_:>8.3f}")

In [ ]:
# 4.3 -- Curvas ROC e Precision-Recall
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

fpr_c, tpr_c, _ = roc_curve(y_real4, proba4)
auc_val = roc_auc_score(y_real4, proba4)
axes[0].plot(fpr_c, tpr_c, 'b-', linewidth=2, label=f'ROC (AUC={auc_val:.3f})')
axes[0].plot([0,1],[0,1],'k--', label='Aleatorio (AUC=0.5)')
axes[0].fill_between(fpr_c, tpr_c, alpha=0.1)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR (Recall)')
axes[0].set_title('Curva ROC'); axes[0].legend()

prec_c, rec_c, _ = precision_recall_curve(y_real4, proba4)
axes[1].plot(rec_c, prec_c, 'g-', linewidth=2)
axes[1].axhline(y_real4.mean(), color='gray', linestyle='--', label=f'Baseline ({y_real4.mean():.2f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall\n(melhor para base desbalanceada)')
axes[1].legend()

threshs = np.arange(0.1, 0.9, 0.05)
precs_t, recs_t, f1s_t = [], [], []
for t in threshs:
    yp = (proba4>=t).astype(int)
    tp_ = ((yp==1)&(y_real4==1)).sum()
    fp_ = ((yp==1)&(y_real4==0)).sum()
    fn_ = ((yp==0)&(y_real4==1)).sum()
    p_  = tp_/(tp_+fp_) if (tp_+fp_)>0 else 0
    r_  = tp_/(tp_+fn_) if (tp_+fn_)>0 else 0
    f_  = 2*p_*r_/(p_+r_) if (p_+r_)>0 else 0
    precs_t.append(p_); recs_t.append(r_); f1s_t.append(f_)

axes[2].plot(threshs, precs_t, 'b-o', markersize=4, label='Precision')
axes[2].plot(threshs, recs_t,  'g-o', markersize=4, label='Recall')
axes[2].plot(threshs, f1s_t,   'r-o', markersize=4, label='F1')
axes[2].set_xlabel('Threshold'); axes[2].set_title('Tradeoff por Threshold')
axes[2].legend()
plt.tight_layout(); plt.show()

print("COMO ESCOLHER O THRESHOLD:")
print("  Fraude: threshold BAIXO -> alto recall (nao deixa fraude passar)")
print("  Credito: threshold ALTO -> alta precision (nao nega credito errado)")
print("  Regra: minimize custo_total = FP*custo_FP + FN*custo_FN")

---
# MODULO 5 -- Cross-Validation e Resampling
## ISLP Capitulo 5

## Por que cross-validation?
Avaliar no mesmo dado que treinou e otimista. Precisamos estimar performance em **dados novos**.

## Tipos de CV:

| Metodo | shuffle | Quando usar |
|--------|---------|-------------|
| `KFold(K, shuffle=False)` | Nao | **PADRAO DA PROVA ITAU** |
| `KFold(K, shuffle=True)` | Sim | Dados i.i.d. sem ordem |
| `StratifiedKFold` | -- | Classificacao desbalanceada |
| `TimeSeriesSplit` | -- | Dados temporais |

**[!] ATENCAO: `neg_mean_squared_error` retorna NEGATIVO -- multiplique por -1!**

**Confirmado com dados reais:**  
Q10 e Q11 da prova = `KFold(n_splits=5, shuffle=False)`


In [ ]:
# 5.1 -- Todos os tipos de CV comparados
np.random.seed(42)
X5, y5 = make_classification(n_samples=600, n_features=12, weights=[0.75,0.25], random_state=42)
modelo5 = LogisticRegression(max_iter=1000)

print("=== COMPARACAO DE ESTRATEGIAS DE CV ===")
configs5 = [
    ('KFold(5, shuffle=False)  <- PADRAO PROVA ITAU', KFold(5, shuffle=False)),
    ('KFold(5, shuffle=True)',                         KFold(5, shuffle=True, random_state=42)),
    ('StratifiedKFold(5)  <- MELHOR para desbalanc.',  StratifiedKFold(5, shuffle=False)),
    ('TimeSeriesSplit(5)  <- para dados temporais',    TimeSeriesSplit(5)),
    ('KFold(10, shuffle=False)',                        KFold(10, shuffle=False)),
]
for nome5, cv5 in configs5:
    res5 = cross_validate(modelo5, X5, y5, cv=cv5, scoring='roc_auc', return_train_score=True)
    print(f"\n{nome5}")
    print(f"  Treino: {res5['train_score'].mean():.4f} | "
          f"Val: {res5['test_score'].mean():.4f} +/- {res5['test_score'].std():.4f} | "
          f"Gap: {res5['train_score'].mean()-res5['test_score'].mean():.4f}")

print()
print("=== ARMADILHA DO SINAL NEGATIVO ===")
res_neg = cross_validate(modelo5, X5, y5, cv=KFold(5, shuffle=False),
                          scoring='neg_log_loss', return_train_score=True)
print(f"  Valor bruto (neg_log_loss): {res_neg['test_score'].mean():.4f}  <- NEGATIVO")
print(f"  x(-1) correto:              {-res_neg['test_score'].mean():.4f}  <- Log Loss real")
print("  SEMPRE: -res['train_score'].mean() e -res['test_score'].mean()")

In [ ]:
# 5.2 -- Learning curves: diagnosticando underfitting e overfitting
X5b, y5b = make_classification(n_samples=1000, n_features=10, random_state=42)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax5, (model5, titulo5) in zip(axes, [
    (DecisionTreeClassifier(max_depth=1),    'Alto Bias (underfitting)\nmax_depth=1'),
    (DecisionTreeClassifier(max_depth=5),    'Equilibrio\nmax_depth=5'),
    (DecisionTreeClassifier(max_depth=None), 'Alta Variancia (overfitting)\nsem poda'),
]):
    tr_sizes, tr_scores, va_scores = learning_curve(
        model5, X5b, y5b, cv=5, scoring='roc_auc',
        train_sizes=np.linspace(0.1,1.0,10), random_state=42)
    ax5.plot(tr_sizes, tr_scores.mean(1), 'b-o', ms=4, label='Treino')
    ax5.plot(tr_sizes, va_scores.mean(1), 'r-o', ms=4, label='Validacao')
    ax5.fill_between(tr_sizes, tr_scores.min(1), tr_scores.max(1), alpha=0.1)
    ax5.fill_between(tr_sizes, va_scores.min(1), va_scores.max(1), alpha=0.1, color='red')
    ax5.set_title(titulo5); ax5.legend(); ax5.set_ylim(0.4, 1.05)
plt.suptitle('Learning Curves -- Diagnostico'); plt.tight_layout(); plt.show()

print("DIAGNOSTICO:")
print("  Underfitting: treino e val convergem para valor BAIXO")
print("    Solucao: modelo mais complexo")
print("  Overfitting: GAP grande (treino alto, val baixo)")
print("    Solucao: regularizacao, mais dados, simplificar")
print("  Equilibrio: treino e val convergem para valor ALTO")

---
# MODULO 6 -- Regularizacao: Ridge, Lasso, Elastic Net
## ISLP Capitulo 6

## As tres formas:
```
Ridge (L2):    Loss = RSS + lambda*sum(beta_j^2)        -> encolhe, NUNCA zera
Lasso (L1):    Loss = RSS + lambda*sum(|beta_j|)        -> pode ZERAR (selecao)
Elastic Net:   Loss = RSS + lambda1*L1 + lambda2*L2     -> melhor dos dois
```

## Confusao critica -- C vs alpha:

| Contexto | Parametro | Regularizacao forte = |
|----------|-----------|----------------------|
| Ridge, Lasso, ElasticNet | `alpha` | alpha GRANDE |
| LogisticRegression, SVM | `C` | **C PEQUENO** (C = 1/alpha) |

## Efeito do lambda (Q14 da prova):
- lambda = 0 -> **regressao linear pura** (sem penalidade)
- lambda -> inf -> coeficientes -> 0 (**NUNCA -> infinito!**)


In [ ]:
# 6.1 -- Ridge vs Lasso vs Elastic Net
np.random.seed(42)
n6, p6 = 400, 25
X6 = np.random.randn(n6, p6)
beta6 = np.zeros(p6); beta6[:6] = [3,-2.5,2,-1.5,1,-0.8]
y6 = X6 @ beta6 + np.random.randn(n6) * 1.2
X6_sc = StandardScaler().fit_transform(X6)
kf6 = KFold(5, shuffle=False)

print("=== RIDGE vs LASSO vs ELASTIC NET ===")
print(f"{'Modelo':<30} {'Coefs=0':>8} {'Coefs!=0':>9} {'MSE val CV':>12}")
print("-" * 63)
for nome6, mod6 in {
    'Linear (sem reg)':           LinearRegression(),
    'Ridge (alpha=1.0)':          Ridge(1.0),
    'Lasso (alpha=0.1)':          Lasso(0.1),
    'ElasticNet (a=0.1, r=0.5)':  ElasticNet(0.1, l1_ratio=0.5),
}.items():
    mod6.fit(X6_sc, y6)
    z6 = (np.abs(mod6.coef_) < 1e-6).sum()
    mse6 = -cross_validate(mod6, X6_sc, y6, cv=kf6,
                            scoring='neg_mean_squared_error')['test_score'].mean()
    print(f"{nome6:<30} {z6:>8} {p6-z6:>9} {mse6:>12.4f}")

print()
print("Ridge: encolhe todos, NUNCA zera -> mantem todas as variaveis")
print("Lasso: zera irrelevantes -> selecao automatica")
print("Elastic Net: melhor dos dois -- util com variaveis correlacionadas")
print()
print("Q14 DA PROVA:")
print("  'lambda=0 em Ridge = regressao linear'     -> VERDADEIRO")
print("  'lambda->inf = coeficientes->infinito'      -> FALSO")
print("  CORRETO: lambda->inf -> coeficientes->0")
print()
print("C no sklearn LogisticRegression (C = 1/alpha):")
for C6 in [0.001, 0.1, 1.0, 10.0]:
    print(f"  C={C6:6.3f} (alpha={1/C6:7.1f}) -> {'regularizacao FORTE' if C6<0.1 else ('moderada' if C6<5 else 'fraca')}")

In [ ]:
# 6.2 -- Caminho dos coeficientes: Lambda vs coefs
alphas6 = np.logspace(-3, 2, 80)
coefs_r6 = [Ridge(a).fit(X6_sc, y6).coef_ for a in alphas6]
coefs_l6 = [Lasso(a, max_iter=5000).fit(X6_sc, y6).coef_ for a in alphas6]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for i6 in range(p6):
    cor6 = 'blue' if i6<6 else 'lightgray'
    lw6  = 2.0    if i6<6 else 0.7
    a6   = 0.9    if i6<6 else 0.25
    axes[0].plot(alphas6, [c[i6] for c in coefs_r6], color=cor6, alpha=a6, linewidth=lw6)
    axes[1].plot(alphas6, [c[i6] for c in coefs_l6], color=cor6, alpha=a6, linewidth=lw6)

for ax6, titulo6 in zip(axes, [
    'RIDGE: coeficientes -> 0 mas NUNCA zerados\n(azul=variaveis reais)',
    'LASSO: coeficientes zerados exatamente\n(azul=variaveis reais)']):
    ax6.set_xscale('log'); ax6.set_xlabel('Alpha (lambda)')
    ax6.set_ylabel('Coeficiente'); ax6.set_title(titulo6)
    ax6.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='alpha=1.0')
    ax6.legend()
plt.tight_layout(); plt.show()

---
# MODULO 7 -- Alem da Linearidade
## ISLP Capitulo 7

| Metodo | Ideia | Quando usar |
|--------|-------|-------------|
| Polynomial Regression | X^2, X^3 | Relacao simples nao-linear |
| Splines | Polinomios por partes | Relacoes complexas locais |
| GAMs | Soma de funcoes suaves | Muitas variaveis nao-lineares |

Cuidado: polinomios de grau alto sao instaveis nas extremidades (Runge's phenomenon)!


In [ ]:
# 7.1 -- Comparando abordagens nao-lineares
np.random.seed(42)
X7 = np.sort(np.random.uniform(0, 10, 200)).reshape(-1, 1)
y7 = np.sin(X7.ravel()) * 3 + 0.3*X7.ravel() + np.random.normal(0, 0.5, 200)
X_plot7 = np.linspace(0, 10, 300).reshape(-1, 1)
kf7 = KFold(5, shuffle=False)

modelos7 = {
    'Linear (grau 1)':           Pipeline([('poly', PolynomialFeatures(1)),  ('lr', LinearRegression())]),
    'Polinomial grau 3':         Pipeline([('poly', PolynomialFeatures(3)),  ('lr', LinearRegression())]),
    'Polinomial grau 10':        Pipeline([('poly', PolynomialFeatures(10)), ('lr', LinearRegression())]),
    'Spline (degree=3, 6 nos)':  Pipeline([('sp', SplineTransformer(degree=3, n_knots=6)), ('lr', LinearRegression())]),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax7, (nome7, mod7) in zip(axes.ravel(), modelos7.items()):
    mod7.fit(X7, y7)
    mse_tr7 = mean_squared_error(y7, mod7.predict(X7))
    mse_va7 = -cross_validate(mod7, X7, y7, cv=kf7,
                               scoring='neg_mean_squared_error')['test_score'].mean()
    ax7.scatter(X7, y7, alpha=0.4, s=15, color='gray')
    ax7.plot(X_plot7, mod7.predict(X_plot7), 'r-', linewidth=2.5)
    ax7.set_title(f'{nome7}\nMSE treino:{mse_tr7:.3f} | val:{mse_va7:.3f}')
plt.tight_layout(); plt.show()

print("Splines: mais estaveis que polinomios de alto grau")
print("Polinomio alto: MSE treino baixo, mas instavel nas bordas")

---
# MODULO 8 -- Arvores, Random Forest, XGBoost
## ISLP Capitulo 8

## Arvores de Decisao
- Criterio de divisao: Gini ou Entropia (classificacao), MSE (regressao)
- Profundidade maxima sem poda: **ceil(log2(n))**
  - n=10M -> log2(10.000.000) = 23.25 -> **profundidade 24** (Q24!)

## Random Forest vs XGBoost -- DIFERENCAS CRITICAS:

| Caracteristica | Random Forest | XGBoost |
|----------------|--------------|---------|
| Construcao | Paralela | Sequencial |
| **learning_rate** | **NAO TEM** | **Tem** |
| Early stopping | Nao | Sim |
| Overfitting via | max_depth alto | lr alto + muitas arvores |
| Reg. L1/L2 | Nao nativa | Built-in |

**Q25 da prova:** 'taxa de aprendizado causa overfitting em RF' -> ERRADO!  
Random Forest NAO tem taxa de aprendizado!


In [ ]:
# 8.1 -- Arvores: profundidade e Log Loss
np.random.seed(42)
X8, y8 = make_classification(n_samples=1000, n_features=12, n_informative=6, random_state=42)
kf8  = KFold(5,  shuffle=False)
kf8b = KFold(10, shuffle=False)

print("=== ARVORE: EFEITO DA PROFUNDIDADE (criterion=entropy) ===")
print(f"{'max_depth':>12} | {'Treino LL':>10} | {'Val LL':>10} | Diagnostico")
print("-" * 65)
for max_d in [None, 1, 3, 5, 10, 20]:
    arv = DecisionTreeClassifier(criterion='entropy', max_depth=max_d, random_state=42)
    res = cross_validate(arv, X8, y8, cv=kf8, scoring='neg_log_loss', return_train_score=True)
    tr8 = -res['train_score'].mean(); va8 = -res['test_score'].mean()
    diag = '<- overfitting total' if max_d is None else ('<- underfitting' if max_d<=2 else ('Equilibrio' if max_d==5 else ''))
    print(f"{str(max_d):>12} | {tr8:>10.4f} | {va8:>10.4f} | {diag}")

print()
print("Q20 DA PROVA: arvore sem poda (entropy, 10 folds):")
arv_prova = DecisionTreeClassifier(criterion='entropy')
res_p = cross_validate(arv_prova, X8, y8, cv=kf8b, scoring='neg_log_loss', return_train_score=True)
print(f"  Treino LL: {-res_p['train_score'].mean():.4f} <- overfitting: memoriza tudo")
print(f"  Val LL:    {-res_p['test_score'].mean():.4f}")
print("  Log Loss E valida para classificacao -- candidato errou Q20!")
print()
prof = math.ceil(math.log2(10_000_000))
print(f"Q24 DA PROVA: log2(10.000.000) = {math.log2(10_000_000):.2f} -> profundidade {prof}")

In [ ]:
# 8.2 -- XGBoost: corrigindo o overfitting do Safra
np.random.seed(42)
X8b, y8b = make_classification(n_samples=1200, n_features=15, n_informative=8, random_state=42)
X_tr8, X_te8, y_tr8, y_te8 = train_test_split(X8b, y8b, test_size=0.2, random_state=42)

print("=== O QUE CAUSOU O OVERFITTING NO SAFRA -- E COMO CORRIGIR ===")
print()
for n_est, lr, depth, lam, desc in [
    (500, 0.5, 12, 0.0, 'MAL configurado (como no Safra)'),
    (200, 0.1,  6, 1.0, 'Moderado'),
    (500, 0.05, 4, 1.0, 'BEM configurado (com early stopping)'),
]:
    use_es = 'BEM' in desc
    m8 = XGBClassifier(n_estimators=n_est, learning_rate=lr, max_depth=depth,
                        reg_lambda=lam, eval_metric='logloss',
                        early_stopping_rounds=30 if use_es else None,
                        verbosity=0, random_state=42)
    if use_es:
        m8.fit(X_tr8, y_tr8, eval_set=[(X_te8, y_te8)], verbose=False)
        n_used = m8.best_iteration
    else:
        m8.fit(X_tr8, y_tr8); n_used = n_est

    auc_tr = roc_auc_score(y_tr8, m8.predict_proba(X_tr8)[:,1])
    auc_te = roc_auc_score(y_te8, m8.predict_proba(X_te8)[:,1])
    print(f"{desc}")
    print(f"  lr={lr}, depth={depth}, lambda={lam} | Arvores usadas: {n_used}")
    print(f"  AUC treino: {auc_tr:.4f} | AUC teste: {auc_te:.4f} | Gap: {auc_tr-auc_te:.4f}")
    print()

print("COMO CORRIGIR OVERFITTING NO XGBOOST:")
print("  1. Reduza learning_rate (0.01-0.1)")
print("  2. Use early_stopping_rounds (para quando val piora)")
print("  3. Reduza max_depth (3-6 e suficiente)")
print("  4. Aumente reg_lambda (L2) e reg_alpha (L1)")
print("  5. Adicione subsample e colsample_bytree (0.7-0.9)")
print()
print("DIFERENCA CRITICA:")
print("  Random Forest: NAO tem learning_rate. Overfitting via max_depth.")
print("  XGBoost: TEM learning_rate. Overfitting via lr alto + muitas arvores.")

---
# MODULO 9 -- Support Vector Machines
## ISLP Capitulo 9

## Regras que nao erram:
- **SEMPRE normalizar** antes do SVM (usa distancias)
- **C pequeno = regularizacao forte** (C = 1/alpha)
- **Gamma alto = overfitting** no kernel RBF (raio pequeno)
- **SVM nao produz probabilidades nativas** -- usa Platt Scaling com `probability=True`
- **Logistica:** probabilidade calibrada por construcao

## Q16 da prova: alto Gamma -> overfitting
Cada ponto influencia regiao MUITO pequena -> memoriza o treino

## Q29 da prova:
- 'SVM nao precisa normalizar' -> **FALSO**
- 'SVM pode atribuir probabilidade' -> **VERDADEIRO** (com probability=True)


In [ ]:
# 9.1 -- SVM: normalizacao, C, Gamma e probabilidade
np.random.seed(42)
X9, y9 = make_classification(n_samples=600, n_features=10, random_state=42)
kf9 = KFold(5, shuffle=False)

print("=== 1. NORMALIZACAO E OBRIGATORIA NO SVM ===")
res_sem = cross_validate(SVC(kernel='rbf', C=1.0, probability=True),
                          X9, y9, cv=kf9, scoring='roc_auc')
res_com = cross_validate(Pipeline([('sc', StandardScaler()),
                                    ('svm', SVC(kernel='rbf', C=1.0, probability=True))]),
                          X9, y9, cv=kf9, scoring='roc_auc')
print(f"  Sem normalizar: AUC = {res_sem['test_score'].mean():.4f}")
print(f"  Com normalizar: AUC = {res_com['test_score'].mean():.4f}")
print("  -> Q29: 'nao e necessario normalizar' = FALSO")

print()
print("=== 2. EFEITO DO PARAMETRO C ===")
print(f"{'C':>8} | {'Treino':>8} | {'Val':>8} | Diagnostico")
for C9 in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    pipe9 = Pipeline([('sc', StandardScaler()), ('svm', SVC(kernel='rbf', C=C9, probability=True))])
    res9 = cross_validate(pipe9, X9, y9, cv=kf9, scoring='roc_auc', return_train_score=True)
    d9 = 'underfitting' if C9<0.01 else ('overfitting' if C9>10 else 'OK')
    print(f"{C9:>8.3f} | {res9['train_score'].mean():>8.4f} | {res9['test_score'].mean():>8.4f} | {d9}")

print()
print("=== 3. EFEITO DO GAMMA (kernel RBF) ===")
print(f"{'Gamma':>10} | {'Treino':>8} | {'Val':>8} | Diagnostico")
for g9 in [0.001, 0.01, 0.1, 1.0, 10.0]:
    pipe9g = Pipeline([('sc', StandardScaler()), ('svm', SVC(kernel='rbf', C=1.0, gamma=g9, probability=True))])
    res9g = cross_validate(pipe9g, X9, y9, cv=kf9, scoring='roc_auc', return_train_score=True)
    d9g = 'raio grande->underfit' if g9<0.01 else ('raio pequeno->overfit' if g9>1 else 'OK')
    print(f"{g9:>10.3f} | {res9g['train_score'].mean():>8.4f} | {res9g['test_score'].mean():>8.4f} | {d9g}")

print()
print("4. SVM E PROBABILIDADE:")
print("   probability=True usa Platt Scaling internamente")
print("   SVM NAO produz probabilidade nativa -- diferente da logistica")
print("   Logistica: calibrada por construcao")
print("   Q27 da prova: arvore NAO tem probabilidade calibrada -- Logistica sim")

In [ ]:
# 9.2 -- Reproduzindo Q10 e Q11 da prova
print("=== REPRODUCAO Q10 E Q11 DA PROVA ===")
print("Padrao confirmado: KFold(K, shuffle=False)")
print()
kf5_sim = KFold(5, shuffle=False)

if os.path.exists('regressao_Q1.csv'):
    df10 = pd.read_csv('regressao_Q1.csv')
    X10, y10 = df10.drop('target',axis=1), df10['target']
    r10 = cross_validate(ElasticNet(alpha=1.0, l1_ratio=0.01), X10, y10,
                          cv=kf5_sim, scoring='neg_mean_squared_error', return_train_score=True)
    print(f"Q10 -- Elastic Net (alpha=1.0, l1_ratio=0.01):")
    print(f"  Treino MSE: {-r10['train_score'].mean():.4f} | Val MSE: {-r10['test_score'].mean():.4f}")
    print(f"  Gabarito:  ~0.2686                           ~0.2693")
else:
    print("Coloque regressao_Q1.csv na pasta para reproduzir Q10")

print()
if os.path.exists('regressao_Q2.csv'):
    df11 = pd.read_csv('regressao_Q2.csv')
    X11, y11 = df11.drop('target',axis=1), df11['target']
    r11 = cross_validate(SVR(kernel='linear', C=0.001), X11, y11,
                          cv=kf5_sim, scoring='neg_mean_squared_error', return_train_score=True)
    print(f"Q11 -- SVR (kernel=linear, C=0.001):")
    print(f"  Treino MSE: {-r11['train_score'].mean():.0f} | Val MSE: {-r11['test_score'].mean():.0f}")
    print(f"  Gabarito:  ~20199                   ~20207")
else:
    print("Coloque regressao_Q2.csv na pasta para reproduzir Q11")

---
# MODULO 10 -- Redes Neurais e Funcoes de Ativacao
## ISLP Capitulo 10

## Funcoes de ativacao -- TUDO que a prova cobra:

| Funcao | Formula | Range | Negativo? |
|--------|---------|-------|-----------|
| **ReLU** | max(0, x) | [0, inf) | **NUNCA** |
| **Leaky ReLU** | max(ax, x) | (-inf, inf) | **PODE** |
| **Sigmoid** | 1/(1+e^-x) | (0, 1) | **NUNCA** |
| **Tanh** | tanh(x) | (-1, 1) | **PODE** |
| **Linear** | x | (-inf, inf) | Pode |
| **Softmax** | e^xi/sum(e^xj) | (0,1) soma=1 | Nunca |

**Q17 da prova:** qual NAO pode gerar -0.001?  
-> ReLU: max(0,x) nunca negativo

**Q28 da prova:** rede com ativacoes lineares em TODAS as camadas  
-> Composicao de lineares = transformacao linear. Profundidade nao ajuda!


In [ ]:
# 10.1 -- Funcoes de ativacao com visualizacao
x_ativ = np.linspace(-5, 5, 300)

funcoes_ativ = {
    'ReLU  max(0,x)\nNUNCA negativo':           lambda x: np.maximum(0, x),
    'Leaky ReLU  0.01x para x<0\nPODE negativo': lambda x: np.where(x>0, x, 0.01*x),
    'Sigmoid  (0,1)\nNUNCA negativo':            lambda x: 1/(1+np.exp(-x)),
    'Tanh  (-1,1)\nPODE negativo':              lambda x: np.tanh(x),
    'Linear  f(x)=x\nRede colapsa se so esta':  lambda x: x,
    'Swish  x*sigmoid(x)\nLevemente negativo':  lambda x: x/(1+np.exp(-x)),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, (nome, func) in zip(axes.ravel(), funcoes_ativ.items()):
    y_f = func(x_ativ)
    ax.plot(x_ativ, y_f, linewidth=2.5, color='steelblue')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(nome, fontsize=10); ax.set_ylim(-2, 2)
plt.suptitle('Funcoes de Ativacao -- Guia Completo', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

print("Q17 DA PROVA: qual NAO pode gerar saida de -0.001?")
print("  ReLU:       max(0,x) -> NUNCA negativo [resposta correta]")
print("  Leaky ReLU: 0.01x para x<0 -> PODE gerar -0.001")
print("  Sigmoid:    entre 0 e 1 -> NUNCA negativo")
print("  Tanh:       entre -1 e 1 -> PODE gerar negativo")
print()
print("Q28 DA PROVA: rede com ativacoes lineares em todas as camadas:")
print("  Linear(Linear(x)) = Linear(x)")
print("  100 camadas lineares = 1 camada linear")
print("  Profundidade SEM nao-linearidade = inutil")
print("  Isso e por que ReLU, Sigmoid e Tanh sao essenciais")

---
# MODULO 11 -- PCA + Clustering
## ISLP Capitulo 12

## PCA
- Reduz dimensionalidade mantendo maxima variancia
- Transforma p variaveis correlacionadas em k componentes nao-correlacionados

## Clustering:

| Algoritmo | K necessario? | Forma dos clusters | Sensivel a escala |
|-----------|--------------|-------------------|-------------------|
| K-means | Sim | Esferica | **Sim -- normalizar!** |
| Hierarquico | Nao | Qualquer | **Sim** |
| DBSCAN | Nao | Qualquer | **Sim** |

**Q36 da prova:** 'clustering e invariante a normalizacao' -> **FALSO!**

## Linkages:

| Linkage | Distancia entre clusters | Sensivel a outliers |
|---------|--------------------------|---------------------|
| **Single** | Minima entre pontos | Muito (Q32!) |
| **Complete** | Maxima entre pontos | Menos (Q32!) |
| **Average** | Media par-a-par (NAO e centroide! -- Q31!) | Moderado |
| **Ward** | Minimiza variancia intra-cluster | Moderado |


In [ ]:
# 11.1 -- PCA
np.random.seed(42)
n11 = 300
grupos11 = np.random.choice([0,1,2], n11, p=[0.4,0.3,0.3])
centros11 = np.array([[3,3,0,0,0,0,0,0,0,0],[-3,-3,0,0,0,0,0,0,0,0],[0,0,3,-3,0,0,0,0,0,0]])
X11 = np.array([centros11[g] + np.random.normal(0,1,10) for g in grupos11])
X11_sc = StandardScaler().fit_transform(X11)

pca11 = PCA().fit(X11_sc)
X11_2d = pca11.transform(X11_sc)[:,:2]
var_exp = pca11.explained_variance_ratio_
var_cum = np.cumsum(var_exp)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(range(1, len(var_exp)+1), var_exp*100, color='steelblue', label='Individual')
axes[0].plot(range(1, len(var_exp)+1), var_cum*100, 'ro-', ms=5, label='Acumulada')
axes[0].axhline(80, color='green', linestyle='--', label='80% variancia')
axes[0].set_title('Scree Plot'); axes[0].legend()

for g in [0,1,2]:
    mask = grupos11==g
    axes[1].scatter(X11_2d[mask,0], X11_2d[mask,1], alpha=0.6, label=f'Grupo {g}')
axes[1].set_xlabel(f'PC1 ({var_exp[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({var_exp[1]*100:.1f}% var)')
axes[1].set_title('Projecao 2D via PCA -- estrutura oculta revelada')
axes[1].legend()

loadings = pd.Series(pca11.components_[0], index=[f'X{i}' for i in range(10)])
loadings.abs().sort_values().plot(kind='barh', ax=axes[2], color='coral')
axes[2].set_title('Loadings PC1')
plt.tight_layout(); plt.show()

n_comp_80 = np.searchsorted(var_cum, 0.80)+1
print(f"Para >= 80% variancia: {n_comp_80} componentes de 10")

In [ ]:
# 11.2 -- Clustering: K-means, hierarquico e todos os linkages
np.random.seed(42)
n_cl = 80
clusters = np.vstack([np.random.normal([2,2],0.4,(n_cl,2)), np.random.normal([8,8],0.4,(n_cl,2)),
                      np.random.normal([2,8],0.4,(n_cl,2)), np.random.normal([8,2],0.4,(n_cl,2))])

# K-means: cotovelo e silhouette
inertias, silhs = [], []
for k in range(2,9):
    km = KMeans(n_clusters=k, n_init=15, random_state=42)
    lbl = km.fit_predict(clusters)
    inertias.append(km.inertia_)
    silhs.append(silhouette_score(clusters, lbl))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes[0,0].plot(range(2,9), inertias, 'bo-')
axes[0,0].axvline(4, color='red', linestyle='--', label='K=4')
axes[0,0].set_title('Metodo do Cotovelo'); axes[0,0].legend()

axes[0,1].plot(range(2,9), silhs, 'go-')
axes[0,1].axvline(4, color='red', linestyle='--', label='K=4')
axes[0,1].set_title('Silhouette Score'); axes[0,1].legend()

lbl4 = KMeans(4, n_init=15, random_state=42).fit_predict(clusters)
for k in range(4):
    axes[0,2].scatter(clusters[lbl4==k,0], clusters[lbl4==k,1], alpha=0.7, label=f'C{k}')
axes[0,2].set_title('K-means (K=4)\nSEMPRE normalizar antes!'); axes[0,2].legend(fontsize=8)

for ax, method, desc in zip(
    [axes[1,0], axes[1,1], axes[1,2]],
    ['single', 'complete', 'ward'],
    ['Single: minima distancia\n(sensivel a outliers -- Q32)',
     'Complete: maxima distancia\n(menos sensivel -- Q32)',
     'Ward: minimiza variancia\n(clusters equilibrados)']
):
    Z = linkage(clusters, method=method)
    dendrogram(Z, ax=ax, no_labels=True, color_threshold=0.7*max(Z[:,2]))
    ax.set_title(desc)
plt.tight_layout(); plt.show()

print("RESUMO DOS LINKAGES:")
print("  single:   minima distancia, sensivel a outliers")
print("  complete: maxima distancia, menos sensivel (Q32 -- correto!)")
print("  average:  media par-a-par -- NAO e centroide (Q31 -- candidato errou!)")
print("  ward:     minimiza variancia, clusters equilibrados")
print()
print("Q36 DA PROVA: 'clustering e invariante a normalizacao' -> FALSO!")
print("  K-means usa distancia Euclidiana -> variavel com maior escala DOMINA")

In [ ]:
# 11.3 -- Q30 da prova: Single Linkage para 4 grupos
if os.path.exists('agrupamento.csv'):
    df30 = pd.read_csv('agrupamento.csv')
    Z30  = linkage(df30.values, method='single')
    print("=== Q30 DA PROVA -- Single Linkage, qual limiar gera 4 grupos? ===")
    for d in sorted(set(Z30[:,2])):
        ng = len(set(fcluster(Z30, d, criterion='distance')))
        if 3 <= ng <= 6:
            print(f"  Limiar = {d:.4f} -> {ng} grupos")
    print()
    print("Para 4 grupos: limiar entre 2.21 e 4.63")
    print("Candidato marcou '0.5 < d < 2.2' -> ERRADO (gera 5 grupos, nao 4)")
    print("Resposta correta: faixa que comeca acima de 2.21")
else:
    print("Coloque agrupamento.csv na pasta para reproduzir Q30")

---
# MODULO 12 -- MLflow: Ciclo de Vida do Modelo

## Por que MLflow?
Sem tracking: voce treina 20 versoes, nao sabe qual parametro usou,  
nao consegue reproduzir, nao sabe qual versao esta em producao.

## Os 4 componentes:

| Componente | O que faz |
|-----------|-----------|
| **Tracking** | Registra experimentos: params, metricas, artefatos |
| **Models** | Serializa e padroniza modelos |
| **Model Registry** | Versiona modelos (Staging -> Production) |
| **Projects** | Empacota codigo com dependencias |

## O que o Itau vai perguntar na entrevista:
- 'Como voce gerencia versoes de modelo?' -> MLflow Tracking + Registry
- 'Como voce sabe se o modelo degradou?' -> Monitoring de drift (nao e MLflow nativo)


In [ ]:
# 12.1 -- MLflow tracking completo
np.random.seed(42)
X12, y12 = make_classification(n_samples=1000, n_features=15, n_informative=8, random_state=42)
X_tr12, X_te12, y_tr12, y_te12 = train_test_split(X12, y12, test_size=0.2, random_state=42)
kf12 = KFold(5, shuffle=False)

mlflow.set_experiment('guia_definitivo_itau')

modelos12 = [
    {'nome': 'LogReg_baseline',
     'modelo': Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(C=1.0, max_iter=1000))]),
     'params': {'tipo': 'LogisticRegression', 'C': 1.0}},
    {'nome': 'RF_balanced',
     'modelo': RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
     'params': {'tipo': 'RandomForest', 'n_trees': 200, 'max_depth': 8}},
    {'nome': 'XGB_early_stop',
     'modelo': XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=4,
                              reg_lambda=1.0, verbosity=0, random_state=42),
     'params': {'tipo': 'XGBoost', 'lr': 0.05, 'depth': 4}},
]

resultados12 = []
for cfg in modelos12:
    with mlflow.start_run(run_name=cfg['nome']):
        mlflow.log_params(cfg['params'])
        res12 = cross_validate(cfg['modelo'], X_tr12, y_tr12, cv=kf12,
                                scoring='roc_auc', return_train_score=True)
        auc_tr = res12['train_score'].mean()
        auc_va = res12['test_score'].mean()
        cfg['modelo'].fit(X_tr12, y_tr12)
        probas12 = cfg['modelo'].predict_proba(X_te12)[:,1]
        auc_te = roc_auc_score(y_te12, probas12)
        ll12   = log_loss(y_te12, probas12)
        mlflow.log_metric('auc_treino', round(auc_tr,4))
        mlflow.log_metric('auc_val',    round(auc_va,4))
        mlflow.log_metric('auc_teste',  round(auc_te,4))
        mlflow.log_metric('log_loss',   round(ll12,4))
        mlflow.log_metric('gap',        round(auc_tr-auc_va,4))
        mlflow.sklearn.log_model(cfg['modelo'], 'modelo')
        resultados12.append({'Modelo': cfg['nome'], 'AUC Tr': round(auc_tr,4),
                              'AUC Val': round(auc_va,4), 'AUC Teste': round(auc_te,4),
                              'LogLoss': round(ll12,4), 'Gap': round(auc_tr-auc_va,4)})

print("RESULTADOS COMPARATIVOS:")
print(pd.DataFrame(resultados12).to_string(index=False))
print()
print("Para visualizar no MLflow UI:")
print("  mlflow ui  (no terminal)")
print("  Acesse: http://localhost:5000")

---
# MODULO 13 -- Simulado Completo da Prova

Coloque os arquivos CSV na mesma pasta do notebook:
- `regressao_Q1.csv` -> Q10
- `regressao_Q2.csv` -> Q11
- `classificacao_Q1.csv` -> Q20
- `classificacao_Q2.csv` -> Q21
- `agrupamento.csv` -> Q30


In [ ]:
# 13.1 -- Simulado com gabaritos
kf5s  = KFold(5,  shuffle=False)
kf10s = KFold(10, shuffle=False)

print("=" * 65)
print("SIMULADO -- PROVA DE 2019 COM DADOS REAIS")
print("Padrao: KFold(K, shuffle=False) -- confirmado!")
print("=" * 65)

# Q10
if os.path.exists('regressao_Q1.csv'):
    df = pd.read_csv('regressao_Q1.csv')
    X, y = df.drop('target',axis=1), df['target']
    r = cross_validate(ElasticNet(alpha=1.0, l1_ratio=0.01), X, y, cv=kf5s,
                        scoring='neg_mean_squared_error', return_train_score=True)
    print(f"\nQ10 -- Elastic Net (alpha=1.0, l1_ratio=0.01):")
    print(f"  Treino MSE: {-r['train_score'].mean():.4f}  (gabarito ~0.2686)")
    print(f"  Val MSE:   {-r['test_score'].mean():.4f}  (gabarito ~0.2693)")

# Q11
if os.path.exists('regressao_Q2.csv'):
    df = pd.read_csv('regressao_Q2.csv')
    X, y = df.drop('target',axis=1), df['target']
    r = cross_validate(SVR(kernel='linear', C=0.001), X, y, cv=kf5s,
                        scoring='neg_mean_squared_error', return_train_score=True)
    print(f"\nQ11 -- SVR (kernel=linear, C=0.001):")
    print(f"  Treino MSE: {-r['train_score'].mean():.0f}  (gabarito ~20199)")
    print(f"  Val MSE:   {-r['test_score'].mean():.0f}  (gabarito ~20207)")

# Q20
if os.path.exists('classificacao_Q1.csv'):
    df = pd.read_csv('classificacao_Q1.csv')
    X, y = df.drop('target',axis=1), df['target']
    r = cross_validate(DecisionTreeClassifier(criterion='entropy'), X, y, cv=kf10s,
                        scoring='neg_log_loss', return_train_score=True)
    print(f"\nQ20 -- Arvore sem poda (entropy, 10 folds):")
    print(f"  Treino LL: {-r['train_score'].mean():.4f}  <- overfitting total")
    print(f"  Val LL:    {-r['test_score'].mean():.4f}")
    print(f"  Candidato errou! Log Loss E valida para classificacao")

# Q21
if os.path.exists('classificacao_Q2.csv'):
    df = pd.read_csv('classificacao_Q2.csv')
    X, y = df.drop('target',axis=1), df['target']
    r = cross_validate(LogisticRegression(C=0.1, max_iter=1000), X, y, cv=kf10s,
                        scoring='roc_auc', return_train_score=True)
    print(f"\nQ21 -- Logistica (C=0.1, L2, 10 folds):")
    print(f"  Treino AUC: {r['train_score'].mean():.4f}")
    print(f"  Val AUC:   {r['test_score'].mean():.4f}")
    print(f"  Candidato errou! Marcou 0.9/0.5 -- modelo bem regularizado nao overfita")

# Q30
if os.path.exists('agrupamento.csv'):
    df = pd.read_csv('agrupamento.csv')
    Z = linkage(df.values, method='single')
    print(f"\nQ30 -- Single Linkage:")
    for d in sorted(set(Z[:,2])):
        ng = len(set(fcluster(Z, d, criterion='distance')))
        if 3 <= ng <= 6:
            print(f"  Limiar={d:.4f} -> {ng} grupos")
    print("  Para 4 grupos: limiar entre 2.21 e 4.63")
    print("  Candidato marcou '0.5<d<2.2' -> ERRADO")

---
# MODULO 14 -- Cola Rapida e Treino de Velocidade

## Quando normalizar:
| Normalizar? | Algoritmos |
|-------------|------------|
| **SEMPRE** | SVM, KNN, Ridge, Lasso, ElasticNet, K-means |
| **NUNCA precisa** | Arvores, Random Forest, XGBoost |

## Metricas por problema:
| Problema | Metricas corretas | NUNCA use |
|----------|-------------------|-----------|
| Regressao | RMSE, MAE, R^2 | MAE para classificacao |
| Classificacao | ROC-AUC, F1, Precision, Recall, Log Loss | Acuracia com base desbalanc. |
| Qualquer | -- | MAPE com y_real=0 |

## Padrao CV da prova Itau:
```python
KFold(n_splits=K, shuffle=False)
-res['test_score'].mean()  # SEMPRE multiplicar por -1
```

## Calibracao de probabilidade:
- **Logistica:** calibrada por construcao
- **RF, Arvores:** NAO calibradas
- **SVM:** Platt Scaling com probability=True

## Linkages:
- **single:** minima distancia, sensivel a outliers
- **complete:** maxima distancia, menos sensivel
- **average:** media par-a-par (NAO e centroide -- Q31!)
- **ward:** minimiza variancia, clusters equilibrados

## Funcoes de ativacao:
- **ReLU:** >= 0, NUNCA negativo
- **Leaky ReLU:** PODE ser negativo
- **Sigmoid:** entre 0 e 1, NUNCA negativo
- **Linear em todas as camadas:** rede colapsa para transformacao linear!


In [ ]:
# 14.1 -- Treino de velocidade: 25 questoes
qa = [
    ("Qual a formula do Bias-Variance Tradeoff?",
     "Erro total = Bias^2 + Variancia + Erro irredutivel (epsilon)"),
    ("lambda=0 em Ridge faz o que?",
     "Elimina a penalidade -> modelo vira regressao linear pura (OLS)"),
    ("lambda->inf em Ridge: coefs vao para?",
     "-> 0. NUNCA -> infinito. (Q14 da prova!)"),
    ("R^2 pode diminuir com mais variaveis?",
     "NAO! R^2 NUNCA diminui. R^2 ajustado PODE."),
    ("Por que nao usar acuracia com base desbalanceada?",
     "Modelo idiota (prever sempre a maioria) tem acuracia alta sem discriminar."),
    ("P-valor: definicao exata",
     "P(observar resultado >= este | H0 verdadeira). NAO e P(H0 verdadeira)."),
    ("Qual CV usar para dados temporais?",
     "TimeSeriesSplit -- treina no passado, valida no futuro."),
    ("Padrao de CV da prova Itau?",
     "KFold(n_splits=K, shuffle=False). Confirmado com Q10 e Q11."),
    ("Como corrigir neg_mean_squared_error?",
     "Multiplicar por -1: -res['test_score'].mean()"),
    ("SVM precisa de normalizacao?",
     "SEMPRE. Usa distancia Euclidiana. (Q29 da prova!)"),
    ("C=0.001 em LogisticRegression = forte ou fraca?",
     "FORTE. C = 1/alpha. C pequeno = alpha grande = regularizacao forte."),
    ("Diferenca Lasso vs Ridge na selecao?",
     "Lasso (L1): ZERA coefs -> selecao automatica. Ridge (L2): encolhe mas nunca zera."),
    ("Qual funcao de ativacao NUNCA gera saida negativa?",
     "ReLU: max(0,x) e Sigmoid: (0,1). Leaky ReLU e Tanh PODEM ser negativos."),
    ("Por que rede com ativacoes lineares falha?",
     "Linear(Linear(x)) = Linear(x). Profundidade nao adiciona capacidade."),
    ("Random Forest tem learning_rate?",
     "NAO. Isso e XGBoost/GBM. Overfitting em RF vem de max_depth alto."),
    ("Como corrigir overfitting no XGBoost?",
     "early_stopping + lr baixo + max_depth menor + reg_lambda."),
    ("K-means precisa de normalizacao?",
     "SEMPRE. Usa distancia Euclidiana -- variavel com maior escala domina."),
    ("Average linkage usa qual distancia?",
     "Media das distancias par-a-par. NAO e centroide. (Q31 da prova!)"),
    ("Qual linkage e MENOS sensivel a outliers?",
     "Complete linkage -- usa distancia maxima. (Q32 da prova!)"),
    ("AUC=0.5 significa?",
     "Modelo aleatorio -- nao discrimina melhor que chance."),
    ("AUC > 0.95 em banco real -> suspeitar de?",
     "Data leakage -- informacao do futuro ou do target vazou para as features."),
    ("Log Loss e valida para classificacao?",
     "SIM. Mede qualidade das probabilidades. (Candidato errou Q20!)"),
    ("MAPE quando y_real=0?",
     "INDEFINIDO -- divisao por zero. (Q13 da prova!)"),
    ("Logistica e calibrada? E Random Forest?",
     "Logistica: calibrada por construcao. RF: NAO calibrada."),
    ("Profundidade maxima arvore sem restricao com n=10M?",
     "ceil(log2(10.000.000)) = ceil(23.25) = 24. (Q24 da prova!)"),
]

print("TREINO DE VELOCIDADE -- 25 QUESTOES")
print("Cubra as respostas e responda mentalmente antes de ver")
print("Meta: cada questao em < 30 segundos")
print("=" * 70)
for i, (pergunta, resposta) in enumerate(qa, 1):
    print(f"\nQ{i:02d}: {pergunta}")
    print(f"  -> {resposta}")

In [ ]:
# 14.2 -- Checklist final pre-prova
checklist = {
    "Estatistica": [
        "Sei a definicao exata de p-valor",
        "Sei a diferenca entre erro tipo 1 e tipo 2",
        "Sei quando usar pearson vs spearman",
        "Entendo que correlacao != causalidade",
        "Sei qual distribuicao assume cada modelo",
    ],
    "Regressao Linear": [
        "Sei interpretar beta com 'mantendo os demais constantes'",
        "Sei que R^2 NUNCA diminui, R^2 adj pode",
        "Sei analisar residuos (QQ plot, homocedasticidade)",
        "Sei que MAPE e indefinido com y_real=0",
        "Sei que sem intercepto: media residuos != 0",
    ],
    "Logistica e Metricas": [
        "Sei calcular Odds Ratio: e^beta",
        "Sei escolher threshold por custo dos erros",
        "Sei calcular P, R, F1, AUC manualmente",
        "Sei que MAE e de regressao, nao classificacao",
        "Sei que Log Loss e valida para classificacao",
    ],
    "Cross-Validation": [
        "Sei que padrao Itau e KFold(K, shuffle=False)",
        "Sei multiplicar neg_* por -1",
        "Sei usar TimeSeriesSplit para dados temporais",
        "Sei usar StratifiedKFold para base desbalanceada",
        "Sei interpretar learning curves",
    ],
    "Regularizacao": [
        "Sei que lambda=0 em Ridge = regressao linear",
        "Sei que lambda->inf: coefs->0 (nunca infinito!)",
        "Sei que Lasso zera coefs, Ridge nao",
        "Sei que C = 1/alpha na logistica",
        "Sei quando usar ElasticNet",
    ],
    "Arvores e Ensemble": [
        "Sei calcular profundidade max: ceil(log2(n))",
        "Sei que arvore sem poda: LL treino = 0",
        "Sei que RF NAO tem learning_rate",
        "Sei usar early_stopping no XGBoost",
        "Sei que RF nao overfita com mais arvores",
    ],
    "SVM": [
        "Sei que SVM SEMPRE precisa normalizar",
        "Sei que C pequeno = regularizacao forte",
        "Sei que Gamma alto = overfitting",
        "Sei que probability=True usa Platt Scaling",
        "Sei reproduzir Q10 e Q11 com KFold sem shuffle",
    ],
    "Clustering": [
        "Sei usar cotovelo e silhouette para escolher K",
        "Sei que K-means SEMPRE precisa normalizar",
        "Sei distinguir os 4 linkages",
        "Sei que average = media par-a-par (nao centroide)",
        "Sei reproduzir Q30 (single linkage, limiar 4 grupos)",
    ],
    "Redes Neurais": [
        "Sei que ReLU e Sigmoid NUNCA geram negativos",
        "Sei que Leaky ReLU PODE gerar negativos",
        "Sei que rede so linear = transformacao linear",
        "Sei quando usar deep learning (nao tabular simples)",
        "Sei o que e vanishing gradient",
    ],
    "MLflow": [
        "Sei logar params, metrics e modelos",
        "Sei iniciar mlflow ui no terminal",
        "Sei que MLflow nao faz monitoring de drift nativo",
        "Sei o que e Model Registry (Staging -> Production)",
        "Sei comparar runs no MLflow UI",
    ],
}

total = sum(len(v) for v in checklist.values())
print(f"CHECKLIST FINAL -- {total} itens para dominar antes da prova")
print("=" * 60)
for modulo, itens in checklist.items():
    print(f"\n[Modulo] {modulo}:")
    for item in itens:
        print(f"  [ ] {item}")

print()
print("ESTRATEGIA PARA OS ULTIMOS DIAS:")
print("  Dia -3: resolver Q10, Q11, Q20, Q21, Q30 do zero sem consultar")
print("  Dia -2: treino de velocidade -- 25 questoes em 12 minutos")
print("  Dia -1: revisar apenas os itens NAO marcados do checklist")
print("  Dia da prova: ler cada enunciado 2x antes de marcar")

---
# MODULO BONUS -- Exercicios Completos por Tema

Resolva cada exercicio SEM olhar o gabarito. So revele depois.


In [ ]:
# EXERCICIOS MODULO 1 -- Estatistica
print("="*65)
print("EXERCICIOS -- ESTATISTICA BASICA")
print("="*65)
print("""
E1: p-valor = 0.03. O que isso significa (escolha a correta)?
  (a) Ha 3% de chance do efeito ser zero
  (b) Ha 97% de chance do efeito ser real
  (c) Se H0 fosse verdade, esse resultado ocorreria 3% das vezes
  (d) O resultado e significativo com alpha=0.01

E2: Volume de transacoes PJ: media=R$800k, mediana=R$200k.
  Qual metrica reportar ao gestor e por que?

E3: Correlacao de 0.82 entre 'dias de atraso' e 'churn'.
  Posso concluir que atraso CAUSA churn? Como investigaria?

E4: No modelo de deteccao de fraude, qual erro e mais grave?
  Erro Tipo 1 (falso positivo) ou Tipo 2 (falso negativo)?

E5: Dados de renda seguem Normal ou Log-normal? Por que?
""")
print("GABARITOS:")
print("""
E1: (c) -- definicao CORRETA de p-valor.
    (a) e (b) sao erros classicos que derrubam candidatos.

E2: MEDIANA (R$200k). Media >> Mediana indica assimetria.
    Poucos clientes de alto volume distorcem a media.
    Mediana representa o cliente tipico.

E3: NAO. Correlacao nao implica causalidade. Pode haver confundidor:
    dificuldade financeira causa TANTO atrasos QUANTO churn.
    Para investigar: experimento randomizado, DiD ou analise de mediacao.

E4: Erro Tipo 2 (falso negativo) = fraude passa despercebida.
    Prejuizo direto e alto. Erro Tipo 1 = bloqueia transacao legitima,
    irritante mas recuperavel. Na maioria dos contextos de fraude: FN >> FP.

E5: Log-normal. Renda e sempre positiva, assimetrica a direita.
    Normal permitiria valores negativos (impossivel para renda).
    Log-normal: sempre positiva, cauda pesada a direita.
""")

In [ ]:
# EXERCICIOS MODULO 3 -- Regressao Linear
print("="*65)
print("EXERCICIOS -- REGRESSAO LINEAR")
print("="*65)
print("""
E1: Coeficiente de 'n_contatos' = -0.42 no modelo de NPS.
    Interprete para o gestor de atendimento.

E2: Modelo A: R^2=0.73 com 4 vars. Modelo B: R^2=0.74 com 12 vars.
    Qual escolher? Como confirmar sua escolha?

E3: Voce adiciona variavel 'ruido_aleatorio' ao modelo.
    O que acontece com R^2? E com R^2 ajustado?

E4: QQ plot mostra pontos nas extremidades saindo da diagonal.
    O que indica? Invalida o modelo?

E5: y_pred=[10,20,0,50] e y_true=[10,20,0,50]. MAPE = 0?
""")
print("GABARITOS:")
print("""
E1: 'Cada contato adicional necessario para resolver o problema
    reduz o NPS em media 0.42 pontos, mantendo tempo de espera,
    tempo de resolucao e nivel de credito constantes.'
    Insight: resolucao no primeiro contato e critica para satisfacao.

E2: Modelo A. R^2 sempre aumenta com mais variaveis -- mesmo com ruido.
    Verificar R^2 ajustado: se B nao melhorar R^2 adj, escolha A.
    Principio da parcimonia: modelo mais simples com mesma capacidade.

E3: R^2 SOBE (ou fica igual) -- nunca diminui.
    R^2 ajustado PODE diminuir -- penaliza a variavel irrelevante.
    Por isso use R^2 ajustado para comparar modelos.

E4: Residuos com caudas pesadas (leptocurtica). Possivel outliers.
    NAO invalida completamente -- com n grande, pelo Teorema Central
    do Limite, a inferencia ainda e aproximadamente valida.
    Verificar: remover outliers? Transformar Y?

E5: NAO. MAPE e INDEFINIDO porque y_true contem 0 (terceiro elemento).
    Divisao por zero! Nunca use MAPE com zeros.
    Use MAE ou RMSE nesses casos.
""")

In [ ]:
# EXERCICIOS MODULO 4 -- Classificacao
import numpy as np
from sklearn.metrics import roc_auc_score, log_loss

print("="*65)
print("EXERCICIOS -- CLASSIFICACAO E METRICAS")
print("="*65)
print("""
E1 (calculo manual):
  TP=90, TN=800, FP=40, FN=70
  Calcule: Acuracia, Precision, Recall, F1

E2: beta de 'atraso_15dias' = 0.75 na logistica.
    Qual o Odds Ratio? Interprete para o gestor de risco.

E3: AUC = 0.98 no teste de modelo de fraude.
    O que voce investigaria ANTES de colocar em producao?

E4: Modelo com Precision=0.95 e Recall=0.25.
    Em qual cenario esse modelo seria util?

E5: Por que Log Loss E valida para classificacao?
    (Candidato errou Q20 da prova!)
""")

print("GABARITOS:")
total_e1 = 90+800+40+70
acc_e1 = (90+800)/total_e1
prec_e1 = 90/(90+40)
rec_e1 = 90/(90+70)
f1_e1 = 2*prec_e1*rec_e1/(prec_e1+rec_e1)
print(f"""
E1: Total = {total_e1}
    Acuracia  = (90+800)/{total_e1} = {acc_e1:.4f}
    Precision = 90/(90+40)          = {prec_e1:.4f}
    Recall    = 90/(90+70)          = {rec_e1:.4f}
    F1        = 2*P*R/(P+R)         = {f1_e1:.4f}

E2: Odds Ratio = e^0.75 = {np.exp(0.75):.3f}
    'Clientes com atraso > 15 dias tem odds de inadimplencia
    {np.exp(0.75):.1f}x maiores, mantendo demais fatores constantes.'
    Ou: 'Atraso aumenta as odds de inadimplencia em {(np.exp(0.75)-1)*100:.0f}%.'

E3: DATA LEAKAGE. AUC=0.98 e suspeita alta.
    Verificar: alguma feature usa informacao do futuro ou do target?
    Testar em janela temporal fora do treino (time-based split).
    Verificar feature importance: alguma variavel domina inesperadamente?

E4: Util quando a ACAO e muito cara.
    Precision=0.95: de 100 alertas, 95 sao fraudadores reais (poucos FP).
    Recall=0.25: so pega 25% das fraudes (perde muitas).
    Cenario: ligacao de retencao custosa -- age so nos casos mais seguros.

E5: Log Loss mede a qualidade das PROBABILIDADES, nao so da classificacao.
    Formula: -mean(y*log(p) + (1-y)*log(1-p))
    Modelo que preve P=0.51 para positivo e punido mais do que
    modelo que preve P=0.99 -- mesmo que ambos classifiquem correto.
    E mais informativa que acuracia e captura calibracao do modelo.
""")

In [ ]:
# EXERCICIOS MODULO 5 -- Cross-Validation
print("="*65)
print("EXERCICIOS -- CROSS-VALIDATION")
print("="*65)
print("""
E1: Colega implementou assim -- identifique o erro:
    scaler = StandardScaler().fit(X)     # fit no X completo!
    X_sc = scaler.transform(X)
    scores = cross_val_score(model, X_sc, y, cv=5)

E2: cross_validate retornou test_score=[-0.43,-0.51,-0.47,-0.49,-0.45].
    scoring='neg_log_loss'. Qual e o Log Loss medio?

E3: Dois modelos:
    A: AUC = 0.82 +/- 0.02  |  B: AUC = 0.85 +/- 0.14
    Qual escolher para producao?

E4: Dados de transacoes Jan/22 a Dez/23. Qual CV usar e por que?

E5: Por que o padrao da prova do Itau e KFold(shuffle=False)?
""")
print("GABARITOS:")
import numpy as np
scores_e2 = np.array([-0.43,-0.51,-0.47,-0.49,-0.45])
print(f"""
E1: DATA LEAKAGE no preprocessamento!
    O scaler foi fitado em X COMPLETO -- inclui o dado de validacao.
    Media e desvio do scaler 'vazam' info do teste para o treino.
    CORRETO: usar Pipeline -- scaler dentro do CV:
    Pipeline([('sc', StandardScaler()), ('model', model)])

E2: Log Loss = -(-0.43-0.51-0.47-0.49-0.45)/5
           = {-scores_e2.mean():.4f}
    SEMPRE multiplique neg_* por -1.

E3: Modelo A (AUC=0.82 +/- 0.02).
    Modelo B tem AUC media maior, mas desvio de 0.14 e enorme.
    Em producao, performance varia muito. Modelo A e muito mais estavel.
    Prefira estabilidade quando AUC e similar.

E4: TimeSeriesSplit.
    Dados temporais tem dependencia serial -- nao sao i.i.d.
    KFold aleatorio usaria dados futuros para prever passado.
    TimeSeriesSplit: treina no passado, valida no futuro.

E5: Confirmado empiricamente com os dados reais da prova:
    Q10 (Elastic Net) e Q11 (SVR) reproduzem os gabaritos EXATOS
    com KFold(shuffle=False). TimeSeriesSplit gera numeros diferentes.
    'Validacao cruzada sequencial' na prova = sem shuffle.
""")

In [ ]:
# EXERCICIOS MODULO 6 -- Regularizacao
import numpy as np

print("="*65)
print("EXERCICIOS -- REGULARIZACAO")
print("="*65)
print("""
E1: 50 variaveis, suspeita que apenas ~8 sao relevantes.
    Ridge, Lasso ou Elastic Net? Por que?

E2: LogReg com C=0.001 vs C=1000.
    Qual tem mais regularizacao? Qual tende a overfittar?

E3: Ridge com lambda=0 vs lambda=10000.
    O que acontece com os coeficientes em cada caso?

E4: 'tempo_espera' e 'tempo_total' com r=0.96 (alta correlacao).
    Usando Lasso: o que acontece? Como Elastic Net seria diferente?

E5: Qual parametro controla regularizacao no sklearn?
    LogisticRegression vs Ridge vs SVM -- qual e a confusao classica?
""")
print("GABARITOS:")
print(f"""
E1: Lasso (ou Elastic Net se houver multicolinearidade).
    Lasso ZERA coefs de variaveis irrelevantes automaticamente.
    Ridge manteria todas as 50 -- nao faz selecao.
    Elastic Net: melhor se as ~8 vars relevantes sao correlacionadas.

E2: C=0.001: MAIS regularizacao (alpha=1000, penalidade forte)
    C=1000:   MENOS regularizacao (alpha=0.001, penalidade fraca)
    C=0.001: mais bias, tende a underfit (modelo muito simples)
    C=1000:  menos bias, mais variancia, tende a overfittar

E3: lambda=0: sem penalidade -> OLS puro (regressao linear classica)
    lambda=10000: penalidade MUITO forte -> coeficientes -> 0
    NUNCA -> infinito! (Erro classico da Q14 da prova)

E4: Lasso: escolhe ARBITRARIAMENTE uma das duas, zera a outra.
    Resultado instavel: pequena mudanca pode trocar qual e escolhida.
    Elastic Net: por causa do L2, MANTEM ambas com coefs menores.
    Distribui o efeito entre variaveis correlacionadas. Mais estavel.

E5: Confusao CRITICA:
    Ridge/Lasso/ElasticNet: parametro = alpha. Regularizacao forte = alpha GRANDE.
    LogisticRegression/SVM: parametro = C. Regularizacao forte = C PEQUENO.
    C = 1/alpha. Sentido INVERSO!
    C=0.001 em LogReg = alpha=1000 em Ridge (ambos: regularizacao forte)
""")

---
# RESUMO DO ISLP POR CAPITULO

Condensado para estudo rapido antes da prova.


In [ ]:
# Resumo ISLP
print("""
CAPITULO 1 -- INTRODUCAO
  Machine learning e um conjunto de ferramentas para entender dados.
  Tres exemplos classicos: renda por educacao (regressao), emails spam (classificacao),
  segmentacao de mercado (clustering).

CAPITULO 2 -- APRENDIZADO ESTATISTICO
  Y = f(X) + epsilon. f e a relacao sistematica. epsilon e o erro irredutivel.
  Predicao: quer prever Y. Inferencia: quer entender X->Y.
  Bias-Variance: Erro = Bias^2 + Variancia + Irredutivel.
  Parametrico (assume forma) vs Nao-parametrico (aprende dos dados).

CAPITULO 3 -- REGRESSAO LINEAR
  OLS: minimiza soma dos quadrados dos erros.
  Coeficiente: efeito de Xj mantendo demais constantes.
  R^2 nunca diminui. R^2 ajustado penaliza complexidade.
  Premissas: linearidade, independencia, homocedasticidade, normalidade dos residuos.
  VIF para multicolinearidade. MAPE indefinido com y=0.

CAPITULO 4 -- CLASSIFICACAO
  Por que nao linear para Y binario: preve fora de [0,1].
  Logistica: sigmoide. Log-odds = Xbeta. Odds Ratio = e^beta.
  LDA: assume Normal multivariada por classe. QDA: mesma mas cov diferente.
  Naive Bayes: assume independencia entre features.
  Metricas: P, R, F1, AUC, Log Loss. Threshold afeta P vs R.

CAPITULO 5 -- RESAMPLING
  Validation set: simples mas depende do split.
  LOOCV: bias minimo, variancia alta, custo alto.
  K-fold: compromisso viavel. k=5 ou 10 sao comuns.
  Bootstrap: reamostragem com reposicao, ~63.2% de cada amostra.

CAPITULO 6 -- SELECAO DE MODELO E REGULARIZACAO
  Melhor subset, forward, backward stepwise.
  Ridge: L2, encolhe, nunca zera. Lasso: L1, pode zerar (selecao).
  Elastic Net: mix L1+L2, melhor com multicolinearidade.
  lambda=0 -> OLS. lambda->inf -> coefs->0 (nunca infinito!).

CAPITULO 7 -- ALEM DA LINEARIDADE
  Polynomial: X^2, X^3. Instavel nas bordas com grau alto.
  Step functions: indicadores por faixas.
  Splines: polinomios por partes com suavidade nos nos.
  GAMs: soma de funcoes suaves, cada variavel com sua propria funcao.

CAPITULO 8 -- ARVORES E ENSEMBLE
  Arvore: divide espaco recursivamente. Criterio: Gini/Entropia/MSE.
  Bagging: bootstrap + media. Random Forest: bagging + subset de features.
  Boosting: sequencial, cada arvore corrige a anterior.
  XGBoost: boosting com regularizacao L1/L2, early stopping.
  RF: sem learning rate. XGB: tem learning rate.

CAPITULO 9 -- SVM
  Hiperplano de maxima margem. Support vectors definem a fronteira.
  Soft margin: C controla tolerancia. C pequeno = mais regularizacao.
  Kernel trick: mapeia para dimensao maior sem calcular explicitamente.
  RBF kernel: Gamma controla raio. Gamma alto = modelo complexo.
  SEMPRE normalizar. NAO produz probabilidades nativamente.

CAPITULO 10 -- DEEP LEARNING
  Rede neural: camadas de neuronios com funcoes de ativacao nao-lineares.
  Sem nao-linearidade: rede colapsa para transformacao linear.
  Backpropagation: gradiente da loss em relacao aos pesos.
  Dropout, batch normalization: regularizacao especifica para redes.
  Quando usar: imagem, texto, sequencia, dados enormes.
  Quando NAO usar: tabular estruturado pequeno/medio (XGB supera).

CAPITULO 12 -- APRENDIZADO NAO-SUPERVISIONADO
  PCA: componentes principais, variancia maxima, sem correlacao.
  K-means: minimiza soma distancias ao centroide. SEMPRE normalizar.
  Hierarquico: single/complete/average/ward. Dendrograma.
  DBSCAN: baseado em densidade, detecta outliers, forma arbitraria.
  Avaliacao: silhouette score, metodo do cotovelo.

CAPITULO 13 -- TESTES DE HIPOTESE MULTIPLOS
  Problema: com muitos testes, falsos positivos se acumulam.
  FWER: Bonferroni, Holm (controla taxa de erro familial).
  FDR: Benjamini-Hochberg (controla taxa de descoberta falsa).
  q-valor: versao do p-valor ajustada para multiplos testes.
""")